In [1]:
import kagglehub
import os

# Download your exact project dataset directly
path = kagglehub.dataset_download("ahmedxc4/skin-ds")
print("Dataset downloaded to:", path)
print("Folders inside:", os.listdir(path))


100%|██████████| 9.25G/9.25G [01:40<00:00, 98.9MB/s]

Extracting files...


Dataset downloaded to: /root/.cache/kagglehub/datasets/ahmedxc4/skin-ds/versions/2
Folders inside: ['README.md', 'test', 'train', 'val']


In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, models, transforms
import json, os

# 1. Connect to the GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Training on:", device)

# 2. Image Preprocessing (Matches your Project Report: 64x64 & Normalized)
transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

# 3. Load Train and Validation Datasets
train_dir = os.path.join(path, "train")
val_dir = os.path.join(path, "val")

train_dataset = datasets.ImageFolder(train_dir, transform=transform)
val_dataset = datasets.ImageFolder(val_dir, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False, num_workers=2)

class_names = train_dataset.classes
num_classes = len(class_names)
print(f"Found {num_classes} classes:", class_names)

# Save class names for our Flask app later
with open("class_names.json", "w") as f:
    json.dump(class_names, f)

# 4. Helper Function to Train & Validate a Model
def train_model(model, name, epochs=5):
    model = model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)

    print(f"\n--- Training {name} ---")
    for epoch in range(epochs):
        model.train()
        correct, total = 0, 0
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()

        # Check Validation Accuracy
        model.eval()
        val_correct, val_total = 0, 0
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                _, predicted = outputs.max(1)
                val_total += labels.size(0)
                val_correct += predicted.eq(labels).sum().item()

        print(f"Epoch {epoch+1}/{epochs} | Train Acc: {100.*correct/total:.2f}% | Val Acc: {100.*val_correct/val_total:.2f}%")
    return model

# 5. Build & Train ResNet-18
resnet = models.resnet18(weights="DEFAULT")
resnet.fc = nn.Linear(resnet.fc.in_features, num_classes)
resnet = train_model(resnet, "ResNet-18", epochs=5)

# 6. Build & Train MobileNetV2
mobilenet = models.mobilenet_v2(weights="DEFAULT")
mobilenet.classifier[1] = nn.Linear(mobilenet.classifier[1].in_features, num_classes)
mobilenet = train_model(mobilenet, "MobileNetV2", epochs=5)

# 7. Evaluate Ensemble Model (Averaging Logits)
resnet.eval()
mobilenet.eval()
ens_correct, ens_total = 0, 0
with torch.no_grad():
    for images, labels in val_loader:
        images, labels = images.to(device), labels.to(device)
        out1 = resnet(images)
        out2 = mobilenet(images)
        avg_logits = (out1 + out2) / 2.0  # Ensemble averaging
        _, predicted = avg_logits.max(1)
        ens_total += labels.size(0)
        ens_correct += predicted.eq(labels).sum().item()

print(f"\n✅ Final Ensemble Validation Accuracy: {100.*ens_correct/ens_total:.2f}%")

# 8. Save Both Models Together for Flask
torch.save({
    "resnet": resnet.state_dict(),
    "mobilenet": mobilenet.state_dict(),
    "num_classes": num_classes
}, "ensemble_trained.pth")
print("✅ Saved ensemble_trained.pth and class_names.json!")


Training on: cuda
Found 14 classes: ['Actinic keratoses', 'Basal cell carcinoma', 'Benign keratosis-like lesions', 'Chickenpox', 'Cowpox', 'Dermatofibroma', 'HFMD', 'Healthy', 'Measles', 'Melanocytic nevi', 'Melanoma', 'Monkeypox', 'Squamous cell carcinoma', 'Vascular lesions']
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 166MB/s]



--- Training ResNet-18 ---
Epoch 1/5 | Train Acc: 65.40% | Val Acc: 71.34%
Epoch 2/5 | Train Acc: 73.66% | Val Acc: 73.55%
Epoch 3/5 | Train Acc: 76.89% | Val Acc: 75.16%
Epoch 4/5 | Train Acc: 79.86% | Val Acc: 75.66%
Epoch 5/5 | Train Acc: 83.12% | Val Acc: 73.91%
Downloading: "https://download.pytorch.org/models/mobilenet_v2-7ebf99e0.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v2-7ebf99e0.pth


100%|██████████| 13.6M/13.6M [00:00<00:00, 143MB/s]


--- Training MobileNetV2 ---


Epoch 1/5 | Train Acc: 68.02% | Val Acc: 75.60%
Epoch 2/5 | Train Acc: 76.42% | Val Acc: 75.77%
Epoch 3/5 | Train Acc: 79.33% | Val Acc: 77.05%
Epoch 4/5 | Train Acc: 81.24% | Val Acc: 78.25%
Epoch 5/5 | Train Acc: 83.01% | Val Acc: 79.13%

✅ Final Ensemble Validation Accuracy: 81.07%
✅ Saved ensemble_trained.pth and class_names.json!


In [7]:
import os, json, io
import torch
import torch.nn as nn
from torchvision import models, transforms
from PIL import Image
from flask import Flask, request, render_template
from google.colab import output

app_code = """import json, io
import torch
import torch.nn as nn
from torchvision import models, transforms
from PIL import Image
from flask import Flask, request, render_template

app = Flask(__name__, template_folder="templates")
device = torch.device("cpu")

with open("class_names.json", "r") as f:
    class_names = json.load(f)

checkpoint = torch.load("ensemble_trained.pth", map_location=device)
num_classes = checkpoint["num_classes"]

resnet = models.resnet18()
resnet.fc = nn.Linear(resnet.fc.in_features, num_classes)
resnet.load_state_dict(checkpoint["resnet"])
resnet.eval()

mobilenet = models.mobilenet_v2()
mobilenet.classifier.add_module("1", nn.Linear(1280, num_classes))
mobilenet.load_state_dict(checkpoint["mobilenet"])
mobilenet.eval()

transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

@app.route("/", methods=["GET", "POST"])
def home():
    prediction = None
    top_probs = []
    if request.method == "POST":
        file = request.files["file"]
        img = Image.open(io.BytesIO(file.read())).convert("RGB")
        tensor = transform(img).unsqueeze(0)
        with torch.no_grad():
            avg_logits = (resnet(tensor) + mobilenet(tensor)) / 2.0
            probs = torch.softmax(avg_logits, dim=1)
        top3_prob, top3_idx = torch.topk(probs, 3)
        flat_idx = top3_idx.view(-1).tolist()
        flat_prob = top3_prob.view(-1).tolist()
        prediction = class_names[flat_idx[0]]
        top_probs = [(class_names[i], round(p * 100, 2)) for i, p in zip(flat_idx, flat_prob)]
    return render_template("index.html", prediction=prediction, top_probs=top_probs)

if __name__ == "__main__":
    app.run(port=5000)
"""

with open("Skin-Cancer-Detection-DL/app.py", "w") as f:
    f.write(app_code)

!rm -f Skin-Cancer-Detection-DL.zip
!zip -r Skin-Cancer-Detection-DL.zip Skin-Cancer-Detection-DL
print("✅ Updated Skin-Cancer-Detection-DL.zip!")

colab_run = app_code.replace('template_folder="templates"', 'template_folder="Skin-Cancer-Detection-DL/templates"')
colab_run = colab_run.replace('if __name__ == "__main__":\n    app.run(port=5000)', 'output.serve_kernel_port_as_window(5000)\napp.run(port=5000)')
exec(colab_run)


  adding: Skin-Cancer-Detection-DL/ (stored 0%)
  adding: Skin-Cancer-Detection-DL/README.md (deflated 41%)
  adding: Skin-Cancer-Detection-DL/templates/ (stored 0%)
  adding: Skin-Cancer-Detection-DL/templates/index.html (deflated 54%)
  adding: Skin-Cancer-Detection-DL/app.py (deflated 55%)
  adding: Skin-Cancer-Detection-DL/requirements.txt (deflated 8%)
  adding: Skin-Cancer-Detection-DL/class_names.json (deflated 38%)
✅ Updated Skin-Cancer-Detection-DL.zip!
Try `serve_kernel_port_as_iframe` instead. 


<IPython.core.display.Javascript object>

 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5000
INFO:werkzeug:Press CTRL+C to quit
INFO:werkzeug:127.0.0.1 - - [19/Nov/2026 15:39:33] "GET / HTTP/1.1" 200 -


In [9]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, models, transforms
from sklearn.metrics import classification_report, confusion_matrix
import os, json, shutil, glob

# 1. Load Test Dataset (Unseen images the model has NEVER trained on)
transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

test_dir = os.path.join(path, "test")
test_dataset = datasets.ImageFolder(test_dir, transform=transform)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

with open("class_names.json", "r") as f:
    class_names = json.load(f)

# 2. Load Your Saved Ensemble Model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
checkpoint = torch.load("ensemble_trained.pth", map_location=device)
num_classes = checkpoint["num_classes"]

resnet = models.resnet18()
resnet.fc = nn.Linear(resnet.fc.in_features, num_classes)
resnet.load_state_dict(checkpoint["resnet"])
resnet.to(device).eval()

mobilenet = models.mobilenet_v2()
mobilenet.classifier.add_module("1", nn.Linear(1280, num_classes))
mobilenet.load_state_dict(checkpoint["mobilenet"])
mobilenet.to(device).eval()

# 3. Predict on All Unseen Test Images
all_preds, all_labels = [], []
with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        avg_logits = (resnet(images) + mobilenet(images)) / 2.0
        _, predicted = avg_logits.max(1)
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.numpy())

# 4. Print Classification Report (Precision, Recall, F1-Score per disease)
print("📊 CLASSIFICATION REPORT ON UNSEEN TEST DATA:")
print(classification_report(all_labels, all_preds, target_names=class_names))

# 5. Export 4 Real Unseen Test Images so you can test your Flask Web App
os.makedirs("real_test_samples", exist_ok=True)
for cls in ["Melanoma", "Healthy", "Basal cell carcinoma", "Monkeypox"]:
    files = glob.glob(os.path.join(test_dir, cls, "*"))
    if files:
      shutil.copy(files.pop(0), f"real_test_samples/TEST_{cls}.jpg")

!zip -r real_test_samples.zip real_test_samples
print("✅ Created real_test_samples.zip! Download it to test your Flask web app.")


📊 CLASSIFICATION REPORT ON UNSEEN TEST DATA:
                               precision    recall  f1-score   support

            Actinic keratoses       0.46      0.38      0.41        88
         Basal cell carcinoma       0.72      0.78      0.75       333
Benign keratosis-like lesions       0.49      0.55      0.52       263
                   Chickenpox       0.92      0.98      0.95       113
                       Cowpox       1.00      0.96      0.98        99
               Dermatofibroma       0.44      0.28      0.34        25
                         HFMD       0.95      0.96      0.95       242
                      Healthy       0.99      0.96      0.97       171
                      Measles       0.99      0.94      0.96        83
             Melanocytic nevi       0.84      0.90      0.87      1288
                     Melanoma       0.67      0.52      0.58       453
                    Monkeypox       0.97      0.97      0.97       426
      Squamous cell carcinoma  